# ParametricNAM Training

Select a GPU runtime. Put this repository and the exported Capture `dataset/` folder on Drive. Keep all files inside the exported folder together. Pick a fresh output directory for each dataset, and reuse that directory to continue the same run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

PROJECT = Path('/content/drive/MyDrive/ParametricNAM-Training')
DATASET_NAME = 'CaptureDataset'  # Folder on MyDrive, e.g. 'VOX'
DATASET = Path('/content/drive/MyDrive') / DATASET_NAME
OUTPUT = Path('/content/drive/MyDrive/ParametricNAM-runs/amp-v1')
TOTAL_STEPS = 20000
BATCH_SIZE = 12


In [ ]:
from importlib.metadata import PackageNotFoundError, version

constraints = Path('/content/pnam-torch-constraints.txt')
pins = []
for package in ('torch', 'torchaudio', 'torchvision'):
    try:
        pins.append(f'{package}=={version(package)}')
    except PackageNotFoundError:
        pass
constraints.write_text('\n'.join(pins) + '\n')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
    '-r', str(PROJECT / 'requirements-colab.txt')
], check=True)

import torch
assert torch.cuda.is_available(), 'Switch this notebook to a GPU runtime.'


In [ ]:
print(f'Checking Drive folder: {DATASET}', flush=True)
candidates = (DATASET, DATASET / 'dataset')
required_files = ('manifest.json', 'profile.json')
dataset_dir = next((folder for folder in candidates
                    if all((folder / name).is_file() for name in required_files)), None)
if dataset_dir is None:
    raise FileNotFoundError(f'No manifest.json and profile.json together in {candidates[0]} or {candidates[1]}. '
                            'Upload the full Capture export and check DATASET_NAME.')
print(f'Using dataset: {dataset_dir}', flush=True)

command = [
    sys.executable, '-u', str(PROJECT / 'run_training.py'),
    '--manifest', str(dataset_dir / 'manifest.json'),
    '--profile', str(dataset_dir / 'profile.json'),
    '--out', str(OUTPUT), '--scratch', '/content',
    '--steps', str(TOTAL_STEPS), '--batch-size', str(BATCH_SIZE), '--device', 'cuda',
]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding='utf-8', errors='replace') as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
if status != 0:
    raise RuntimeError(f'Training exited with code {status}. See the trainer output above.')
print(OUTPUT / 'model.pnam')


For a short smoke run, set a total of 2 steps and a batch size of 1 in a separate output directory. Once the main run completes, download `model.pnam` from its output directory and open it in ParametricNAM.